# SAM Pipeline Smoke Test (Dense Matrix)

Use this notebook from the repository root or open it directly in VSCode/Jupyter with the `.venv` Python kernel. It starts with a small in-memory test of the dense builder, then provides optional cells for Databricks and local output validation.

The model convention is `Z[i, j] = flow from supplier node i to buyer node j` (rows = origin, columns = destination). The SAM is stored as a **dense** `float64` matrix; small and negative values are preserved (no sparsification or thresholding).

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parents[0]

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(PROJECT_ROOT)

In [ ]:
import tempfile

import numpy as np
import pandas as pd

from climate_risk_io.sam.dense_builder import (
    build_node_index,
    build_nodes,
    compute_output_vector,
    create_dense_memmap,
    write_flows_to_dense_matrix,
)
from climate_risk_io.sam.validators import validate_dense_model_outputs
from config.paths import SAM_PROCESSED_DIR

print("Imports ok")
print("SAM processed dir:", SAM_PROCESSED_DIR)

## 1. Local Dense-Builder Smoke Test

This does not connect to Databricks. It verifies node construction, the dense memory-mapped matrix orientation, and the output vector. Note the synthetic flows include a negative value and a tiny `1e-9` value, both of which must be preserved.

In [ ]:
flows = pd.DataFrame(
    {
        "origin_region": ["R1", "R1", "R2", "R2"],
        "origin_sector": ["S1", "S1", "S2", "S1"],
        "destination_region": ["R2", "R2", "R1", "R2"],
        "destination_sector": ["S2", "S2", "S1", "S1"],
        "flow_value": [1.0, 2.0, -0.5, 1e-9],
    }
)

# Node universe = union of origin and destination (region, sector) pairs.
pairs = pd.concat(
    [
        flows[["origin_region", "origin_sector"]].rename(
            columns={"origin_region": "region_code", "origin_sector": "sector_code"}
        ),
        flows[["destination_region", "destination_sector"]].rename(
            columns={
                "destination_region": "region_code",
                "destination_sector": "sector_code",
            }
        ),
    ],
    ignore_index=True,
)

nodes = build_nodes(pairs)
node_index = build_node_index(nodes)

# Aggregate duplicate (i, j) pairs, then fill a dense memmap matrix.
aggregated = flows.groupby(
    ["origin_region", "origin_sector", "destination_region", "destination_sector"],
    as_index=False,
)["flow_value"].sum()

tmp_dir = Path(tempfile.mkdtemp())
Z = create_dense_memmap(tmp_dir / "z_matrix.npy", len(nodes))
write_flows_to_dense_matrix(Z, aggregated, node_index)
Z.flush()

x = compute_output_vector(Z)
validate_dense_model_outputs(nodes, Z, x)

print("PASS: local dense-builder smoke test")
print("Z shape:", Z.shape)
print("Z sum:", float(Z.sum()))
print("x sum:", float(x.sum()))
print("tiny value 1e-9 preserved:", bool(np.any(np.isclose(np.asarray(Z), 1e-9))))
display(nodes)
display(aggregated)
display(
    pd.DataFrame(
        {
            "node_id": nodes["node_id"],
            "node_label": nodes["node_label"],
            "x_output": x,
        }
    )
)

## 2. Optional Databricks Table Check

Set `RUN_DATABRICKS = True` only when your Databricks Connect profile/session is ready (or the Databricks VSCode extension is running).

In [ ]:
RUN_DATABRICKS = False

if RUN_DATABRICKS:
    from climate_risk_io.sam.databricks_client import read_sam_table
    from climate_risk_io.sam.mappings import SAM_TABLE

    sdf = read_sam_table()
    print("Table:", SAM_TABLE)
    sdf.printSchema()
    display(sdf.limit(10).toPandas())
    print("Rows:", sdf.count())
else:
    print("Skipped Databricks check. Set RUN_DATABRICKS = True to run it.")

## 3. Optional Script Runner

These cells run the same scripts documented in the README. Keep the build off until you have chosen filters in `scripts/build_sam_dense_matrix_from_databricks.py`.

In [ ]:
import subprocess

def run_script(script_name: str):
    result = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / script_name)],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

print(sys.executable)

In [ ]:
RUN_INSPECT_SCRIPT = False
RUN_BUILD_DENSE_SCRIPT = False
RUN_TEST_DENSE_SCRIPT = False

if RUN_INSPECT_SCRIPT:
    run_script("inspect_sam_databricks.py")
if RUN_BUILD_DENSE_SCRIPT:
    run_script("build_sam_dense_matrix_from_databricks.py")
if RUN_TEST_DENSE_SCRIPT:
    run_script("test_sam_dense_matrix.py")

if not any([RUN_INSPECT_SCRIPT, RUN_BUILD_DENSE_SCRIPT, RUN_TEST_DENSE_SCRIPT]):
    print("No scripts selected. Set one or more RUN_* flags to True.")

## 4. Local Output Check

Run this after building the dense matrix. It loads the generated dense artifacts (memory-mapped) and validates matrix dimensions and totals.

In [ ]:
from climate_risk_io.sam.loaders import load_sam_dense_model

required_outputs = [
    SAM_PROCESSED_DIR / "nodes.csv",
    SAM_PROCESSED_DIR / "z_matrix.npy",
    SAM_PROCESSED_DIR / "x_vector.npy",
    SAM_PROCESSED_DIR / "sam_build_report.json",
]

missing = [path for path in required_outputs if not path.exists()]
if missing:
    print("Missing outputs. Run the dense build first:")
    for path in missing:
        print("-", path)
else:
    model = load_sam_dense_model(SAM_PROCESSED_DIR, mmap=True)
    validate_dense_model_outputs(model["nodes"], model["Z"], model["x"])
    print("PASS: processed dense SAM outputs are valid")
    print("Z shape:", model["Z"].shape)
    print("Z sum:", float(model["Z"].sum()))
    print("sum(x):", float(model["x"].sum()))
    display(model["nodes"].head())
    model["report"]